<a href="https://colab.research.google.com/github/jawahirkhaleel/-jawahirkhaleel-/blob/main/pystac_ndvi_anomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install -q pystac-client stackstac

In [33]:
from pystac_client import Client


In [34]:
catalog = Client.open(
    'https://earth-search.aws.element84.com/v1'
)

In [35]:
for collection in catalog.get_collections():
  print(collection.id)

sentinel-2-pre-c1-l2a
cop-dem-glo-30
naip
cop-dem-glo-90
landsat-c2-l2
sentinel-2-l2a
sentinel-2-l1c
sentinel-2-c1-l2a
sentinel-1-grd


In [31]:
roi = [72.90, 33.60, 73.15, 33.75]


In [36]:
search = catalog.search(
    collections = ['sentinel-2-c1-l2a'],
    bbox = roi,
    datetime = '2025'
)

In [37]:
items = search.item_collection()


/usr/local/lib/python3.13/dist-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail']
  warnings.warn(


In [38]:
len(items)

174

In [39]:
items[0].assets.keys()

dict_keys(['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail'])

In [40]:
import stackstac

In [41]:
ds = stackstac.stack(
    items,
    assets = ['red','nir','scl'],
    bounds_latlon= roi,
    epsg = 3857,
    resolution = 10
)

In [42]:
ds_clean = (
    ds.to_dataset('band')
    .drop_dims('band')
    .reset_coords(drop = True)
    .drop_attrs()

)

In [43]:
ds_clean


<xarray.Dataset> Size: 23GB
Dimensions:  (time: 174, y: 2008, x: 2784)
Coordinates:
  * time     (time) datetime64[ns] 1kB 2025-01-04T05:59:40.840000 ... 2025-12...
  * y        (y) float64 16kB 3.995e+06 3.995e+06 ... 3.975e+06 3.975e+06
  * x        (x) float64 22kB 8.115e+06 8.115e+06 ... 8.143e+06 8.143e+06
Data variables:
    red      (time, y, x) float64 8GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
    nir      (time, y, x) float64 8GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
    scl      (time, y, x) float64 8GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>

In [44]:
cloud = ds_clean.scl.isin([3,8,9,10])
veg = ds_clean.scl.isin(4).max('time')

In [45]:
ds_var = ds_clean[['red','nir']].where(cloud == 0)

In [46]:
nir = ds_var['nir']
red = ds_var['red']

In [47]:
ndvi = (nir - red) / (nir + red)

In [48]:
ndvi_monthly = ndvi.resample(time = 'M').mean('time')

/usr/local/lib/python3.13/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [49]:
ndvi_monthly = ndvi_monthly.where(veg == 1)

In [50]:
ndvi_monthly


<xarray.DataArray (time: 12, y: 2008, x: 2784)> Size: 537MB
dask.array<where, shape=(12, 2008, 2784), dtype=float64, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 96B 2025-01-31 2025-02-28 ... 2025-12-31
  * y        (y) float64 16kB 3.995e+06 3.995e+06 ... 3.975e+06 3.975e+06
  * x        (x) float64 22kB 8.115e+06 8.115e+06 ... 8.143e+06 8.143e+06

In [51]:
from dask.diagnostics import ProgressBar


In [52]:
import warnings
warnings.filterwarnings('ignore', category = RuntimeWarning)

In [53]:
import logging
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

In [ ]:
with ProgressBar():
  ndvi_monthly = ndvi_monthly.compute()

[#############################           ] | 74% Completed | 291.52 s

In [ ]:
ndvi_monthly.plot(
    robust = True,
    col = 'time',
    col_wrap = 6,
    cmap = 'RdYlGn'
)

In [ ]:
ndvi_anomaly = ndvi_monthly - ndvi_monthly.mean('time')

In [ ]:
ndvi_anomaly.plot(
    robust = True,
    col = 'time',
    col_wrap = 6
)


In [ ]:
ndvi_diff = ndvi_monthly.diff('time')


In [ ]:
ndvi_diff.plot(
    robust = True,
    col = 'time',
    col_wrap = 6
)